# Tomato Leaf Classification - PyTorch Training Pipeline (Colab Version)

Este notebook executa o treinamento de modelos ResNet-18 do zero (sem pesos pré-treinados) nas fatias do dataset preparado (`subset_1%` até `subset_100%`).

### Requisitos Implementados:
1. **Detecção Universal de Hardware**: CUDA (NVIDIA) -> DirectML (AMD) -> CPU.
2. **Sem Data Augmentation**: Apenas Redimensionamento (224x224) e Normalização.
3. **Múltiplas Execuções (5x por subset)**: Loop sobre 5 sementes (seeds) aleatórias diferentes para medir estabilidade de inicialização.
4. **Métricas de Análise**: Registra a Loss de treino por época, e a Acurácia e Precisão (Macro) no conjunto de teste.
5. **Salvamento Estruturado**: Consolida todas as métricas em um arquivo CSV.

--- 
## 1. Setup / Device Detection

Execute esta célula para importar as dependências necessárias, montar o Google Drive e detectar o hardware disponível (CUDA, DirectML ou CPU).

In [ ]:
# Instala dependências no Colab caso necessário
# !pip install scikit-learn torch-directml -q

import csv
import random
import time
from pathlib import Path
from typing import Dict, List, Tuple

import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
import torchvision
from torchvision.datasets import ImageFolder
from torchvision import transforms
from sklearn.metrics import precision_score

# Montagem do Google Drive no Colab
try:
    from google.colab import drive
    print("Montando Google Drive...")
    drive.mount('/content/drive')
except ImportError:
    print("Ambiente Colab não detectado. Executando localmente.")

def get_device() -> Tuple[torch.device, str]:
    """
    Detecta o hardware de treinamento na ordem de prioridade:
    1º CUDA (NVIDIA) -> 2º DirectML (AMD Windows) -> 3º CPU
    """
    if torch.cuda.is_available():
        return torch.device("cuda"), "NVIDIA/CUDA (GPU)"
    
    try:
        import torch_directml
        if torch_directml.is_available():
            return torch_directml.device(), "AMD/DirectML (GPU via DirectML)"
    except ImportError:
        pass
        
    return torch.device("cpu"), "CPU"

device, hardware_name = get_device()
print("=" * 50)
print(f"Hardware Detectado: {hardware_name}")
print(f"PyTorch Device Object: {device}")
print("=" * 50)

--- 
## 2. Modelo e Métricas

Esta célula define a função para fixar o seed de inicialização, carrega as transformações de imagens (sem data augmentation), instancia o modelo ResNet-18 do zero e configura os métodos de avaliação.

In [ ]:
def set_seed(seed: int):
    """
    Garante reprodutibilidade nas sementes de pesos e amostragem.
    """
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def get_transforms() -> transforms.Compose:
    """
    Sem data augmentation: Apenas redimensionamento para entrada padrão da ResNet (224x224)
    e normalização estatística ImageNet.
    """
    return transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
        transforms.Normalize(
            mean=[0.485, 0.456, 0.406],
            std=[0.229, 0.224, 0.225]
        )
    ])


@torch.no_grad()
def evaluate_model(
    model: nn.Module,
    dataloader: DataLoader,
    device: torch.device
) -> Tuple[float, float]:
    """
    Calcula Acurácia e Precisão (Macro) no conjunto de teste.
    """
    model.eval()
    preds_list = []
    labels_list = []

    for inputs, labels in dataloader:
        inputs = inputs.to(device)
        outputs = model(inputs)
        _, preds = torch.max(outputs, 1)
        
        preds_list.extend(preds.cpu().numpy())
        labels_list.extend(labels.numpy())
        
    preds_arr = np.array(preds_list)
    labels_arr = np.array(labels_list)
    
    correct = (preds_arr == labels_arr).sum()
    accuracy = correct / len(labels_arr) if len(labels_arr) > 0 else 0.0
    
    precision = precision_score(labels_arr, preds_arr, average="macro", zero_division=0.0)
    
    return float(accuracy), float(precision)

--- 
## 3. Loop de Treinamento

Aqui definimos a função de treino de uma época e o loop orquestrador que percorre todos os subsets (`[1, 2, 5, 10, 20, 50, 100]`) e executa o processo 5 vezes (com seeds de 42 a 46) para cada subset.

In [ ]:
def train_one_epoch(
    model: nn.Module,
    dataloader: DataLoader,
    criterion: nn.Module,
    optimizer: optim.Optimizer,
    device: torch.device
) -> float:
    """
    Treina uma única época do modelo e retorna a loss média.
    """
    model.train()
    running_loss = 0.0
    total_samples = 0
    
    for inputs, labels in dataloader:
        inputs = inputs.to(device)
        labels = labels.to(device)
        
        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        
        running_loss += loss.item() * inputs.size(0)
        total_samples += inputs.size(0)
        
    return running_loss / total_samples if total_samples > 0 else 0.0

--- 
## 4. Orquestração e Salvamento

Execute esta célula para rodar o pipeline experimental. Defina abaixo os caminhos relativos ao seu Google Drive ou armazenamento local. O resultado será consolidado em um CSV ao final da execução.

In [ ]:
# --- CONFIGURAÇÕES DO EXPERIMENTO ---
# Altere para caminhos do Google Drive se necessário, ex:
# DATA_DIR = Path("/content/drive/MyDrive/redes_neurais/dataset/preprocessed")
# OUTPUT_CSV = Path("/content/drive/MyDrive/redes_neurais/training_results_colab.csv")

DATA_DIR = Path("../dataset/preprocessed")
OUTPUT_CSV = Path("training_results.csv")

EPOCHS = 10
BATCH_SIZE = 32
LR = 0.001
SUBSETS = [1, 2, 5, 10, 20, 50, 100]
RUNS_PER_SUBSET = 5

seeds = [42 + i for i in range(RUNS_PER_SUBSET)]
img_transforms = get_transforms()

print(f"Orquestrador inicializado no device: {device}")
print(f"Treinando subsets: {SUBSETS} | Épocas: {EPOCHS} | Repetições: {RUNS_PER_SUBSET}")

# Inicializa arquivo CSV de métricas
csv_headers = ["subset", "run_index", "seed", "epoch", "train_loss", "test_accuracy", "test_precision", "epoch_time_sec"]
with open(OUTPUT_CSV, "w", newline="", encoding="utf-8") as f:
    writer = csv.writer(f)
    writer.writerow(csv_headers)

# Loop pelos subsets
for s in SUBSETS:
    subset_name = f"subset_{s}%"
    subset_path = DATA_DIR / subset_name
    
    if not subset_path.exists():
        print(f"\n[AVISO] Pasta {subset_name} não encontrada em {subset_path}. Pulando.")
        continue
        
    print(f"\n\n=== Treinando no {subset_name} ===")
    
    train_dataset = ImageFolder(root=subset_path / "train", transform=img_transforms)
    test_dataset = ImageFolder(root=subset_path / "test", transform=img_transforms)
    
    train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=0)
    test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)
    
    for run_idx, seed in enumerate(seeds):
        print(f"\n  -> Execução {run_idx + 1}/{RUNS_PER_SUBSET} (Semente: {seed})")
        
        # Fixa semente antes de inicializar o modelo para garantir pesos determinísticos
        set_seed(seed)
        
        # Inicializa ResNet-18 sem pesos pré-treinados (do zero)
        model = torchvision.models.resnet18(weights=None, num_classes=10)
        model = model.to(device)
        
        criterion = nn.CrossEntropyLoss()
        optimizer = optim.Adam(model.parameters(), lr=LR)
        
        for epoch in range(1, EPOCHS + 1):
            t_start = time.time()
            
            # Uma época de treino e validação
            loss_treino = train_one_epoch(model, train_loader, criterion, optimizer, device)
            acc_teste, prec_teste = evaluate_model(model, test_loader, device)
            
            t_epoch = time.time() - t_start
            
            print(f"    Época {epoch:02d}/{EPOCHS:02d} | Loss Treino: {loss_treino:.4f} | Acurácia Teste: {acc_teste:.4f} | Precisão Teste: {prec_teste:.4f} | Tempo: {t_epoch:.2f}s")
            
            # Salva no arquivo CSV
            with open(OUTPUT_CSV, "a", newline="", encoding="utf-8") as f:
                writer = csv.writer(f)
                writer.writerow([
                    subset_name,
                    run_idx,
                    seed,
                    epoch,
                    round(loss_treino, 5),
                    round(acc_teste, 5),
                    round(prec_teste, 5),
                    round(t_epoch, 2)
                ])

print(f"\n[SUCESSO] Pipeline finalizado! Métricas consolidadas em: {OUTPUT_CSV.resolve()}")